# CNeuroMod QA — tSNR brain maps

Volumetric montages of average temporal-SNR maps (MNI space), one panel per subject and one per dataset, read from `output_data/tsnr_maps/`. Figures are written to `output_data/figures/tsnr_maps/`.

tSNR is volumetric and QA cares about signal dropout in ventral/orbitofrontal, temporal and subcortical regions, so we render faithful volumetric slices (nilearn) rather than a cortical surface, which would discard subcortex/cerebellum.

In [1]:
import os
from pathlib import Path

import nibabel as nib
import numpy as np
from nilearn import plotting

# Paths are provided by `invoke run-notebooks` as environment variables.
# Figures go in output_data/figures/{FIG_NAME}/ (also the notebook's "already
# ran" sentinel); the tSNR maps live in output_data/tsnr_maps/{dataset}/.
FIG_NAME = "tsnr_maps"
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data"))
TSNR_DIR = OUTPUT_DIR / "tsnr_maps"
FIG_DIR = OUTPUT_DIR / "figures" / FIG_NAME
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Shared display settings so panels are visually comparable.
CMAP = "inferno"
DISPLAY_MODE = "z"       # axial montage — shows ventral/subcortical dropout
N_CUTS = 8
DPI = 120

In [2]:
def dataset_average_maps():
    """{dataset: path} for each per-dataset average map under output_data/tsnr_maps/."""
    maps = {}
    if TSNR_DIR.is_dir():
        for dataset_dir in sorted(p for p in TSNR_DIR.iterdir() if p.is_dir()):
            hits = sorted(dataset_dir.glob(f"{dataset_dir.name}_*_stat-avgtsnr_statmap.nii.gz"))
            if hits:
                maps[dataset_dir.name] = hits[0]
    return maps


def subject_maps(dataset):
    """Sorted per-subject avgtsnr maps for one dataset (excludes the dataset avg)."""
    dataset_dir = TSNR_DIR / dataset
    return sorted(dataset_dir.glob("sub-*_stat-avgtsnr_statmap.nii.gz"))


def robust_vmax(paths):
    """98th percentile of positive tSNR across maps — a shared, outlier-robust ceiling."""
    values = []
    for path in paths:
        data = np.asarray(nib.load(str(path)).dataobj, dtype=np.float32)
        data = data[np.isfinite(data) & (data > 0)]
        if data.size:
            values.append(np.percentile(data, 98))
    return float(np.median(values)) if values else None


dataset_maps = dataset_average_maps()
print(f"found dataset-average maps for: {sorted(dataset_maps)}")
if not dataset_maps:
    print("No tSNR maps found — run `invoke run-tsnr-maps` first (needs data access).")
VMAX = robust_vmax(dataset_maps.values())
print(f"shared vmax = {VMAX}")

found dataset-average maps for: ['floc']
shared vmax = 53.689910888671875


In [3]:
def plot_tsnr(map_path, title, out_path, vmax=None):
    """Axial montage of one tSNR map on the MNI template; saved to out_path."""
    display = plotting.plot_stat_map(
        str(map_path), display_mode=DISPLAY_MODE, cut_coords=N_CUTS,
        cmap=CMAP, vmax=vmax, colorbar=True, black_bg=True,
        title=title, symmetric_cbar=False,
    )
    display.savefig(str(out_path), dpi=DPI)
    display.close()


# One montage per dataset average.
for dataset, path in dataset_maps.items():
    plot_tsnr(path, f"{dataset} — average tSNR",
              FIG_DIR / f"{dataset}_avgtsnr.png", vmax=VMAX)

/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)


In [4]:
# One montage per subject, per dataset.
for dataset in dataset_maps:
    for path in subject_maps(dataset):
        subject = path.name.split("_", 1)[0]  # e.g. sub-01
        plot_tsnr(path, f"{dataset} — {subject} tSNR",
                  FIG_DIR / f"{dataset}_{subject}_avgtsnr.png", vmax=VMAX)

/home/pbellec/git/cneuromod.all.qa_figures/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2599: RuntimeWarning: invalid value encountered in <lambda> (vectorized)
  outputs = ufunc(*args, out=...)
